# MEDREx — Phase 1: Exploratory Data Analysis
**CSC 590 Masters Project | CSUDH Spring 2026**  
**Student:** Kingslee Dominic Savio Velu | **Mentor:** Dr. Bin Tang

---
This notebook explores the TCGA pathology report dataset before any model training.
Goal: understand the data shape, class distribution, text characteristics, and potential issues.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR     = r'c:\Users\kings\OneDrive\Desktop\CSUDH_Kingslee\CSUDH-Spring-2026\CSC-590-MastersProject\MastersProject'
REPORTS_PATH = os.path.join(BASE_DIR, 'DataSet', 'TCGA_Reports.csv', 'TCGA_Reports.csv')
LABELS_PATH  = os.path.join(BASE_DIR, 'DataSet', 'tcga_patient_to_cancer_type.csv')

print('Paths configured.')
print('Reports:', REPORTS_PATH)
print('Labels: ', LABELS_PATH)

## 1. Load Data

In [ ]:
df_reports = pd.read_csv(REPORTS_PATH)
df_labels  = pd.read_csv(LABELS_PATH)

print('Reports shape:', df_reports.shape)
print('Labels shape: ', df_labels.shape)
print()
print('Reports columns:', df_reports.columns.tolist())
print('Labels columns: ', df_labels.columns.tolist())

In [ ]:
df_reports.head(3)

In [ ]:
df_labels.head(3)

## 2. Join Datasets on patient_id

In [ ]:
# Extract patient_id from patient_filename
# Example: 'TCGA-BP-5195.25c0b433-...' -> 'TCGA-BP-5195'
df_reports['patient_id'] = df_reports['patient_filename'].apply(lambda x: x.split('.')[0])

# Merge
df = df_reports.merge(df_labels, on='patient_id', how='inner')

print(f'After join: {len(df):,} rows')
print(f'Reports not matched: {len(df_reports) - len(df)}')
print()
df.head(3)

## 3. Missing Value Check

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Empty text reports ===')
empty = df['text'].apply(lambda x: len(str(x).strip()) == 0).sum()
print(f'Empty text: {empty}')

## 4. Cancer Type Distribution

In [ ]:
CANCER_NAMES = {
    'ACC':'Adrenocortical carcinoma', 'BLCA':'Bladder Urothelial Carcinoma',
    'BRCA':'Breast invasive carcinoma', 'CESC':'Cervical squamous cell carcinoma',
    'CHOL':'Cholangiocarcinoma', 'COAD':'Colon adenocarcinoma',
    'DLBC':'Diffuse Large B-cell Lymphoma', 'ESCA':'Esophageal carcinoma',
    'GBM':'Glioblastoma multiforme', 'HNSC':'Head and Neck squamous cell carcinoma',
    'KICH':'Kidney Chromophobe', 'KIRC':'Kidney renal clear cell carcinoma',
    'KIRP':'Kidney renal papillary cell carcinoma', 'LAML':'Acute Myeloid Leukemia',
    'LGG':'Brain Lower Grade Glioma', 'LIHC':'Liver hepatocellular carcinoma',
    'LUAD':'Lung adenocarcinoma', 'LUSC':'Lung squamous cell carcinoma',
    'MESO':'Mesothelioma', 'OV':'Ovarian serous cystadenocarcinoma',
    'PAAD':'Pancreatic adenocarcinoma', 'PCPG':'Pheochromocytoma and Paraganglioma',
    'PRAD':'Prostate adenocarcinoma', 'READ':'Rectum adenocarcinoma',
    'SARC':'Sarcoma', 'SKCM':'Skin Cutaneous Melanoma',
    'STAD':'Stomach adenocarcinoma', 'TGCT':'Testicular Germ Cell Tumors',
    'THCA':'Thyroid carcinoma', 'THYM':'Thymoma',
    'UCEC':'Uterine Corpus Endometrial Carcinoma', 'UCS':'Uterine Carcinosarcoma',
    'UVM':'Uveal Melanoma'
}

ct_counts = df['cancer_type'].value_counts()
print(f'Total unique cancer types: {len(ct_counts)}')
print()
print(ct_counts.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))

labels = [f"{ct} ({CANCER_NAMES.get(ct, ct)[:30]})" for ct in ct_counts.index]
colors = sns.color_palette('Blues_r', len(ct_counts))

bars = ax.barh(labels, ct_counts.values, color=colors)
ax.set_xlabel('Number of Reports', fontsize=12)
ax.set_title('TCGA Cancer Type Distribution (35 Classes)', fontsize=14, fontweight='bold')
ax.invert_yaxis()

for bar, val in zip(bars, ct_counts.values):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'notebooks', 'cancer_type_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 5. Class Imbalance Analysis

In [ ]:
most_common = ct_counts.idxmax()
least_common = ct_counts.idxmin()
imbalance_ratio = ct_counts.max() / ct_counts.min()

print(f'Most common:   {most_common} — {ct_counts.max()} reports ({CANCER_NAMES.get(most_common,"")})')
print(f'Least common:  {least_common} — {ct_counts.min()} reports ({CANCER_NAMES.get(least_common,"")})')
print(f'Imbalance ratio: {imbalance_ratio:.1f}x')
print()
print('Classes with < 100 reports:')
print(ct_counts[ct_counts < 100])

## 6. Text Length Analysis

In [ ]:
df['word_count']  = df['text'].apply(lambda x: len(str(x).split()))
df['char_count']  = df['text'].apply(lambda x: len(str(x)))

print('=== Word Count Statistics ===')
print(df['word_count'].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['word_count'], bins=60, color='#1a3a5c', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Words per Report')
axes[0].set_ylabel('Number of Reports')
axes[0].set_title('Report Length Distribution')
axes[0].axvline(df['word_count'].median(), color='red', linestyle='--',
                label=f'Median: {int(df["word_count"].median())} words')
axes[0].axvline(512, color='orange', linestyle='--', label='BERT limit (512 tokens)')
axes[0].legend()

# Box plot per cancer type (top 10)
top10 = ct_counts.head(10).index
df_top10 = df[df['cancer_type'].isin(top10)]
df_top10.boxplot(column='word_count', by='cancer_type', ax=axes[1], rot=45)
axes[1].set_title('Word Count by Cancer Type (Top 10)')
axes[1].set_xlabel('Cancer Type')
axes[1].set_ylabel('Words per Report')
plt.suptitle('')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'notebooks', 'text_length_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

pct_over_512 = (df['word_count'] > 512).mean() * 100
print(f'Reports exceeding 512 words (BERT limit): {pct_over_512:.1f}%')

## 7. Sample Reports

In [ ]:
print('=== Sample Reports (first 500 chars) ===\n')
for cancer in ['BRCA', 'GBM', 'LUAD']:
    sample = df[df['cancer_type'] == cancer].iloc[0]
    print(f'--- {cancer} ({CANCER_NAMES.get(cancer,"")}) ---')
    print(str(sample['text'])[:500])
    print()

## 8. Summary

In [ ]:
print('=' * 55)
print('  EDA SUMMARY — TCGA Pathology Reports')
print('=' * 55)
print(f'  Total reports:          {len(df):,}')
print(f'  Cancer types:           {df["cancer_type"].nunique()}')
print(f'  Missing values:         {df.isnull().sum().sum()}')
print(f'  Avg words/report:       {df["word_count"].mean():.0f}')
print(f'  Median words/report:    {df["word_count"].median():.0f}')
print(f'  Reports > 512 words:    {pct_over_512:.1f}% (chunking needed for BERT)')
print(f'  Most common class:      {most_common} ({ct_counts.max()} reports)')
print(f'  Least common class:     {least_common} ({ct_counts.min()} reports)')
print(f'  Class imbalance ratio:  {imbalance_ratio:.1f}x')
print('=' * 55)
print('  Key issue: class imbalance -> use class_weight="balanced"')
print('  Key issue: reports > 512 tokens -> chunking for BERT/RAG')
print('=' * 55)